# Traffic Collision Analysis & Flow Simulation in Toronto Using Cellular Automata Models
## Based on Boddepalli et al. (2026) — Adapted for KSI_converted Dataset

**Dataset:** Toronto KSI (Killed or Seriously Injured) collision records (2006–2024)
**Data source:** `data/toronto-dataset/KSI_converted.csv`

# Section 1: Setup, Data Loading & Cleaning

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import os, warnings; warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, roc_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.cluster import DBSCAN

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150
os.makedirs('outputs/toronto_reference', exist_ok=True)
print('Libraries loaded')

In [ ]:
# Load KSI_converted dataset
ksi = pd.read_csv(r'data/toronto-dataset/KSI_converted.csv', low_memory=False)
print(f'Shape: {ksi.shape}')
print(f'Columns ({len(ksi.columns)}): {list(ksi.columns)}')
print(f'Unique accidents: {ksi["ACCNUM"].nunique()}')
print(f'Years: {sorted(ksi["YEAR"].dropna().unique().astype(int))}')

### Build Accident-Level Aggregation
Collapse per-person records into one row per accident with aggregated features.

In [ ]:
# Accident-level aggregation
event_cols = ['ROAD_CLASS','DISTRICT','LATITUDE','LONGITUDE','TRAFFCTL','VISIBILITY',
              'LIGHT','RDSFCOND','IMPACTYPE','YEAR','DATE','INITDIR','ACCLASS']
acc_list = []
for accnum, grp in ksi.groupby('ACCNUM', sort=False):
    feat = {'ACCNUM': accnum}
    for c in event_cols:
        vals = grp[c].dropna()
        feat[c] = vals.iloc[0] if len(vals) > 0 else (np.nan if c in ['LATITUDE','LONGITUDE'] else '<Null>')
    feat['num_persons'] = len(grp)
    ages = pd.to_numeric(grp['INVAGE'], errors='coerce').dropna()
    feat['min_age_acc'] = ages.min() if len(ages) > 0 else np.nan
    feat['max_age_acc'] = ages.max() if len(ages) > 0 else np.nan
    feat['avg_age_acc'] = ages.mean() if len(ages) > 0 else np.nan
    for col in ['INVTYPE','VEHTYPE','INITDIR','MANOEUVER','DRIVACT','DRIVCOND']:
        feat['n_unique_'+col.lower()] = grp[col].dropna().nunique()
    for col in ['PEDESTRIAN','CYCLIST','AUTOMOBILE','MOTORCYCLE','TRUCK','SPEEDING','AG_DRIV','REDLIGHT']:
        feat[col] = 'Yes' if (grp[col]=='Yes').any() else '<Null>'
    acc_list.append(feat)
acc_df = pd.DataFrame(acc_list)
print(f'Accident-level shape: {acc_df.shape}')
print(f'Total accidents: {len(acc_df)}')

### Per-Person Data with Imputed INJURY

In [ ]:
# Per-person data
person_cols = ['ACCNUM','INJURY','INVAGE','INVTYPE','VEHTYPE','INITDIR','MANOEUVER',
               'DRIVACT','DRIVCOND','HOUR','YEAR']
person_cols = [c for c in person_cols if c in ksi.columns]
persons = ksi[person_cols].copy()
persons['INJURY'] = persons['INJURY'].fillna('Unknown')
persons['target'] = persons['INJURY'].map({'Unknown':0,'Minimal':1,'Minor':2,'Major':3,'Fatal':4}).astype(int)
df = persons.merge(acc_df, on='ACCNUM', how='left')
print(f'Total rows: {len(df)}')
print(f'Unique accidents: {df["ACCNUM"].nunique()}')

In [ ]:
# Severity distribution
sev_map = {'Unknown':0,'Minimal':1,'Minor':2,'Major':3,'Fatal':4}
rev_sev = {v:k for k,v in sev_map.items()}
tnames = [rev_sev[i] for i in range(5)]
print('Severity distribution (per-person, imputed):')
for name in tnames:
    cnt = (df['target'] == sev_map[name]).sum()
    print(f'  {name}: {cnt} ({cnt/len(df)*100:.1f}%)')

### Save Cleaned Accident-Level Data for CA Simulation

In [ ]:
# Save for later use
acc_df.to_csv('data/toronto-dataset/accidents_aggregated.csv', index=False)
print('Saved accident-level data to accidents_aggregated.csv')

# Section 2: Exploratory Data Analysis (EDA)

### 2.1 Collision Trends Over Years

In [ ]:
# Collisions per year
dp = pd.to_datetime(acc_df['DATE'], errors='coerce')
acc_df['Year'] = dp.dt.year
yearly = acc_df['Year'].value_counts().sort_index()
print('Collisions per year:')
print(yearly.to_string())

In [ ]:
# Bar chart: collisions per year
plt.figure(figsize=(12,5))
yearly.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Total Collisions in Toronto (2006–2024)', fontsize=14)
plt.xlabel('Year'); plt.ylabel('Number of Collisions')
plt.xticks(rotation=45); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/toronto_reference/collisions_per_year.png', dpi=150)
plt.show()

### 2.2 Collision Hotspots — Top Neighbourhoods

In [ ]:
# Top neighbourhoods
if 'NEIGHBOURHOOD' in acc_df.columns:
    hoods = acc_df['NEIGHBOURHOOD'].value_counts().head(10)
    plt.figure(figsize=(10,5))
    hoods.plot(kind='barh', color='firebrick', edgecolor='black')
    plt.title('Top 10 Neighbourhoods by Collision Count')
    plt.xlabel('Number of Collisions'); plt.ylabel('Neighbourhood')
    plt.gca().invert_yaxis(); plt.grid(axis='x', alpha=0.3)
    plt.tight_layout(); plt.savefig('outputs/toronto_reference/top_neighbourhoods.png', dpi=150)
    plt.show()
else:
    print('NEIGHBOURHOOD column not found, using DISTRICT instead')
    hoods = acc_df['DISTRICT'].value_counts().head(10)
    plt.figure(figsize=(10,5))
    hoods.plot(kind='barh', color='firebrick', edgecolor='black')
    plt.title('Top 10 Districts by Collision Count')
    plt.xlabel('Number of Collisions'); plt.ylabel('District')
    plt.gca().invert_yaxis(); plt.grid(axis='x', alpha=0.3)
    plt.tight_layout(); plt.savefig('outputs/toronto_reference/top_districts.png', dpi=150)
    plt.show()

### 2.3 Temporal Analysis — Collisions by Hour

In [ ]:
# Collisions by hour (from per-person data)
hourly = persons['HOUR'].dropna().astype(int).value_counts().sort_index()
plt.figure(figsize=(12,5))
hourly.plot(kind='bar', color='mediumseagreen', edgecolor='black')
plt.title('Collisions by Hour of the Day', fontsize=14)
plt.xlabel('Hour (24-hour format)'); plt.ylabel('Number of Collisions')
plt.xticks(rotation=0); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/toronto_reference/collisions_by_hour.png', dpi=150)
plt.show()
print('Peak hours:', hourly.nlargest(5).index.sort_values().tolist())

### 2.4 Driver Behavior Analysis

In [ ]:
# Aggressive driving involvement
if 'AG_DRIV' in acc_df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16,4))
    for ax, col, title, color in zip(axes,
        ['AG_DRIV','SPEEDING','REDLIGHT'],
        ['Aggressive Driving','Speeding','Red Light Running'],
        ['coral','goldenrod','tomato']):
        if col in acc_df.columns:
            cnts = acc_df[col].value_counts()
            cnts.plot(kind='bar', ax=ax, color=color, edgecolor='black')
            ax.set_title(title); ax.set_xlabel('Involvement'); ax.set_ylabel('Count')
            ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.savefig('outputs/toronto_reference/behavior_analysis.png', dpi=150)
    plt.show()

### 2.5 Spatial Clustering with DBSCAN

In [ ]:
# DBSCAN clustering on accident locations
coords = acc_df[['LATITUDE','LONGITUDE']].dropna().values
if len(coords) > 0:
    db = DBSCAN(eps=0.01, min_samples=10)
    clusters = db.fit_predict(coords)
    acc_df_loc = acc_df.dropna(subset=['LATITUDE','LONGITUDE']).copy()
    acc_df_loc['Cluster'] = clusters
    n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
    print(f'DBSCAN found {n_clusters} clusters, noise points: {(clusters==-1).sum()}')

    plt.figure(figsize=(12,8))
    scatter = plt.scatter(acc_df_loc['LONGITUDE'], acc_df_loc['LATITUDE'],
                c=acc_df_loc['Cluster'], cmap='tab20', s=15, alpha=0.6)
    plt.colorbar(scatter, label='Cluster')
    plt.title('DBSCAN Clustering of Collision Locations')
    plt.xlabel('Longitude'); plt.ylabel('Latitude')
    plt.tight_layout(); plt.savefig('outputs/toronto_reference/dbscan_clusters.png', dpi=150)
    plt.show()

### 2.6 Correlation Matrix

In [ ]:
# Correlation of numerical features
num_cols = [c for c in ['Year','HOUR','LATITUDE','LONGITUDE','num_persons',
                         'min_age_acc','avg_age_acc','max_age_acc'] if c in acc_df.columns]
if len(num_cols) > 1:
    corr = acc_df[num_cols].corr()
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
                linewidths=0.5, square=True)
    plt.title('Correlation Matrix of Numerical Features')
    plt.tight_layout(); plt.savefig('outputs/toronto_reference/correlation_matrix.png', dpi=150)
    plt.show()

# Section 3: Cellular Automata Traffic Flow Simulation

Based on the reference: using collision data to derive traffic flow parameters,
direction weights, green time allocation, and delay prediction.

### 3.1 Directional Flow Analysis

In [ ]:
# Directional analysis using INITDIR
dir_counts = acc_df['INITDIR'].value_counts()
print('Direction distribution:')
print(dir_counts)
print()

# Normalized direction weights
dir_weights = dir_counts / dir_counts.sum()
print('Direction weights (normalized):')
for d, w in dir_weights.items():
    print(f'  {d}: {w:.4f} ({w*100:.1f}%)')

In [ ]:
# Bar chart of direction distribution
plt.figure(figsize=(8,5))
dir_counts.plot(kind='bar', color=['steelblue','coral','forestgreen','goldenrod','gray'], edgecolor='black')
plt.title('Collision Distribution by Direction (INITDIR)', fontsize=13)
plt.xlabel('Direction'); plt.ylabel('Number of Collisions')
plt.xticks(rotation=0); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/toronto_reference/direction_distribution.png', dpi=150)
plt.show()

### 3.2 Risk Weight Calculation per Direction

In [ ]:
# Compute risk scores per direction using driver behaviour factors
dir_risk = {}
for direction in ['North','South','East','West']:
    subset = acc_df[acc_df['INITDIR'] == direction]
    risk = {
        'count': len(subset),
        'ag_driv_pct': (subset['AG_DRIV']=='Yes').mean() * 100 if 'AG_DRIV' in subset else 0,
        'speeding_pct': (subset['SPEEDING']=='Yes').mean() * 100 if 'SPEEDING' in subset else 0,
        'redlight_pct': (subset['REDLIGHT']=='Yes').mean() * 100 if 'REDLIGHT' in subset else 0,
        'fatal_pct': (subset['ACCLASS']=='Fatal').mean() * 100 if 'ACCLASS' in subset else 0,
    }
    risk['combined_risk'] = (risk['ag_driv_pct'] * 0.3 + risk['speeding_pct'] * 0.3
                             + risk['redlight_pct'] * 0.2 + risk['fatal_pct'] * 0.2)
    dir_risk[direction] = risk

risk_df = pd.DataFrame(dir_risk).T
print('Risk metrics per direction:')
print(risk_df.round(2))

In [ ]:
# Visualize risk metrics
risk_df[['ag_driv_pct','speeding_pct','redlight_pct','fatal_pct']].plot(
    kind='bar', figsize=(10,5), edgecolor='black')
plt.title('Risk Factors by Direction', fontsize=13)
plt.xlabel('Direction'); plt.ylabel('Percentage (%)')
plt.xticks(rotation=0); plt.grid(axis='y', alpha=0.3)
plt.legend(title='Risk Factor')
plt.tight_layout(); plt.savefig('outputs/toronto_reference/risk_by_direction.png', dpi=150)
plt.show()

### 3.3 Traffic Signal Green Time Allocation

In [ ]:
# Total cycle time and green time allocation
total_cycle = 60  # seconds
capacity_factor = 15  # vehicles per green-second

# Normalized weights based on collision count (higher collisions = more congestion = more green time needed)
dir_norm = dir_weights / dir_weights.sum()

signal_df = pd.DataFrame({
    'direction': dir_norm.index,
    'weight': dir_norm.values,
    'collision_count': dir_counts.values
})
# Green time proportional to directional demand (higher collisions = more vehicles)
signal_df['green_time_s'] = (signal_df['weight'] * total_cycle).round(2)
signal_df['routing_cost'] = 1.0 / (signal_df['weight'] + 0.01)
# Normalize routing cost
signal_df['routing_cost_norm'] = (signal_df['routing_cost'] - signal_df['routing_cost'].min()) / (
    signal_df['routing_cost'].max() - signal_df['routing_cost'].min() + 0.001)
print('Signal Control Parameters:')
print(signal_df.to_string(index=False))

In [ ]:
# Visualize green time allocation
plt.figure(figsize=(8,5))
colors = ['steelblue','coral','forestgreen','goldenrod','gray']
bars = plt.bar(signal_df['direction'], signal_df['green_time_s'],
               color=colors[:len(signal_df)], edgecolor='black')
plt.title('Green Time Allocation per Direction (Total Cycle: 60s)', fontsize=13)
plt.xlabel('Direction'); plt.ylabel('Green Time (seconds)')
for bar, gt in zip(bars, signal_df['green_time_s']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{gt}s', ha='center', va='bottom', fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/toronto_reference/green_time_allocation.png', dpi=150)
plt.show()

### 3.4 Vehicle Routing Simulation

In [ ]:
# Softmax routing
def softmax(x):
    e_x = np.exp(-x)
    return e_x / e_x.sum()

total_vehicles = 1000
routing_probs = softmax(signal_df['routing_cost_norm'].values)
signal_df['vehicles_routed'] = np.round(routing_probs * total_vehicles).astype(int)

# Simulate throughput
throughput_rate = 1.5  # vehicles per second green
signal_df['vehicles_passed'] = (signal_df['green_time_s'] * throughput_rate).round(0).astype(int)
signal_df['overflow'] = np.maximum(0, signal_df['vehicles_routed'] - signal_df['vehicles_passed'])
signal_df['utilization_pct'] = (signal_df['vehicles_passed'] / signal_df['vehicles_routed'] * 100).round(1)

print('Traffic Flow Simulation Results:')
print(signal_df[['direction','green_time_s','vehicles_routed','vehicles_passed','overflow','utilization_pct']].to_string(index=False))

### 3.5 Delay Prediction Model

In [ ]:
# Refined delay model
base_delay = 20  # seconds minimum

def predict_delay(green_time, vehicles_routed, capacity=15, base=20):
    load = vehicles_routed / capacity
    delay = base + (load ** 2) * 50 / max(green_time, 1)
    return max(delay, base)

signal_df['predicted_delay'] = signal_df.apply(
    lambda r: predict_delay(r['green_time_s'], r['vehicles_routed']), axis=1
).round(2)

# Delay reduction compared to base
signal_df['delay_reduction'] = (base_delay - signal_df['predicted_delay']).round(2)

print('Delay Prediction:')
print(signal_df[['direction','green_time_s','vehicles_routed','predicted_delay','delay_reduction']].to_string(index=False))

In [ ]:
# Visualize delay prediction
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].bar(signal_df['direction'], signal_df['predicted_delay'],
            color='salmon', edgecolor='black')
axes[0].set_title('Predicted Delay per Direction (seconds)')
axes[0].set_xlabel('Direction'); axes[0].set_ylabel('Delay (s)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(signal_df['direction'], signal_df['delay_reduction'],
            color='seagreen', edgecolor='black')
axes[1].set_title('Delay Reduction per Direction (seconds)')
axes[1].set_xlabel('Direction'); axes[1].set_ylabel('Delay Reduction (s)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.savefig('outputs/toronto_reference/delay_prediction.png', dpi=150)
plt.show()

### 3.6 Sankey Diagram — Directional Flow

In [ ]:
import plotly.graph_objects as go

# Prepare Sankey data
directions = signal_df['direction'].tolist()
cost_categories = []
for cost in signal_df['routing_cost_norm']:
    if cost <= 0.33: cost_categories.append('Low Cost')
    elif cost <= 0.66: cost_categories.append('Medium Cost')
    else: cost_categories.append('High Cost')

labels = directions + list(set(cost_categories))
label_idx = {l:i for i,l in enumerate(labels)}
src_idx = [label_idx[d] for d in directions]
tgt_idx = [label_idx[c] for c in cost_categories]

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, line=dict(color='black', width=0.5), label=labels),
    link=dict(source=src_idx, target=tgt_idx, value=signal_df['weight'].tolist())
)])
fig.update_layout(title_text='Directional Flows by Routing Cost Category', font_size=12)
fig.write_html('outputs/toronto_reference/sankey_flow.html')
fig.show()

# Section 4: ML — Collision Severity Prediction (Multiclass)

Predict injury severity (Unknown/Minimal/Minor/Major/Fatal) using person-level and accident-level features.

### 4.1 Feature Engineering

In [ ]:
# Feature engineering
dp = pd.to_datetime(df['DATE'], errors='coerce')
df['Year'] = dp.dt.year.fillna(0).astype(int)
df['Month'] = dp.dt.month.fillna(1).astype(int)
df['DayOfWeek'] = dp.dt.day_name()
df['is_weekend'] = (dp.dt.dayofweek >= 5).astype(int)
df['Season'] = df['Month'].map({12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3}).fillna(0).astype(int)
df['is_night'] = df['HOUR'].isin([22,23,0,1,2,3,4,5]).astype(int)

# Ordinal mappings
light_map={'Daylight':1,'Dawn':1.5,'Dusk':1.5,'Dark Artificial':3,'Dark':4}
rdsfc_map={'Dry':1,'Wet':2,'Loose Snow':3,'Slush':3.5,'Packed Snow':4,'Ice':5}
vis_map={'Clear':1,'Cloudy':2,'Rain':3,'Fog':3,'Mist':3,'Snow':4,'Freezing Rain':5,'Drifting Snow':5}
for col,m in [('LIGHT',light_map),('RDSFCOND',rdsfc_map),('VISIBILITY',vis_map)]:
    if col in df.columns: df[col+'_o'] = df[col].map(m).fillna(0)

# Spatial grid cells
lat_c, lon_c = 0.00225, 0.003125
df['g_i'] = ((df['LATITUDE'] - df['LATITUDE'].min()) / lat_c).fillna(0).astype(int)
df['g_j'] = ((df['LONGITUDE'] - df['LONGITUDE'].min()) / lon_c).fillna(0).astype(int)

# Age imputation
if 'INVAGE' in df.columns:
    df['INVAGE'] = pd.to_numeric(df['INVAGE'], errors='coerce').fillna(df['INVAGE'].median())

# Drop leaky / ID columns
for c in ['INJURY','ACCNUM','DATE']:
    if c in df.columns: df.drop(columns=[c], inplace=True)

# Fill remaining NAs
for c in df.select_dtypes(include=['object']).columns:
    m = df[c].mode()
    df[c] = df[c].fillna(m[0] if len(m) > 0 else 'Unknown')
for c in df.select_dtypes(include=['int64','float64']).columns:
    df[c] = df[c].fillna(df[c].median())

print(f'Feature engineering done. Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Encode categorical features
cat_f = [c for c in ['ROAD_CLASS','DISTRICT','TRAFFCTL','IMPACTYPE','INVTYPE','VEHTYPE',
         'INITDIR','MANOEUVER','DRIVACT','DRIVCOND','DayOfWeek','Season'] if c in df.columns]
bool_f = [c for c in ['PEDESTRIAN','CYCLIST','AUTOMOBILE','MOTORCYCLE','TRUCK','SPEEDING','AG_DRIV','REDLIGHT'] if c in df.columns]

for c in cat_f:
    df[c+'_e'] = LabelEncoder().fit_transform(df[c].astype(str))
for c in bool_f:
    df[c+'_b'] = (df[c].astype(str).str.upper() == 'YES').astype(int)

# Accident-level features
acc_feats = ['num_persons','min_age_acc','max_age_acc','avg_age_acc',
             'n_unique_invtype','n_unique_vehtype','n_unique_initdir',
             'n_unique_manoeuver','n_unique_drivact','n_unique_drivcond']
acc_feats = [c for c in acc_feats if c in df.columns]

# Final feature list
f_cols = ([c+'_e' for c in cat_f] + [c+'_b' for c in bool_f] +
          [c+'_o' for c in ['LIGHT','RDSFCOND','VISIBILITY'] if c+'_o' in df.columns] +
          ['INVAGE','Month','Year','is_weekend','is_night','g_i','g_j','LATITUDE','LONGITUDE','HOUR'] + acc_feats)
f_cols = [c for c in f_cols if c in df.columns]
print(f'Total features: {len(f_cols)}')

### 4.2 Train/Test Split (Temporal)

In [ ]:
# Temporal split: 2006-2022 train, 2023-2024 test
tr = df[~df['Year'].isin([2023, 2024])].copy()
te = df[df['Year'].isin([2023, 2024])].copy()
print(f'Train: {len(tr)} rows ({tr["Year"].min()}-{tr["Year"].max()})')
print(f'Test:  {len(te)} rows ({te["Year"].min()}-{te["Year"].max()})')

Xtr = tr[f_cols].fillna(0).values; ytr = tr['target'].values
Xte = te[f_cols].fillna(0).values; yte = te['target'].values

# Scale features
ss = StandardScaler()
Xtr_s = ss.fit_transform(Xtr)
Xte_s = ss.transform(Xte)

print(f'Train class distribution: {np.bincount(ytr)}')
print(f'Test  class distribution: {np.bincount(yte)}')

### 4.3 Random Forest Classifier

In [ ]:
print('Training Random Forest (n=100, d=10)...')
rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(Xtr_s, ytr)
yp_rf = rf.predict(Xte_s)
ypb_rf = rf.predict_proba(Xte_s)
acc_rf = accuracy_score(yte, yp_rf)
f1_rf = f1_score(yte, yp_rf, average='macro')
print(f'RF Accuracy: {acc_rf:.4f}, Macro F1: {f1_rf:.4f}')
print()
print('Classification Report:')
print(classification_report(yte, yp_rf, target_names=tnames, zero_division=0))

### 4.4 XGBoost Classifier

In [ ]:
print('Training XGBoost (lr=0.1, d=6, n=150)...')
xgb = XGBClassifier(learning_rate=0.1, max_depth=6, n_estimators=150,
                    random_state=42, eval_metric='mlogloss',
                    objective='multi:softprob', num_class=5)
xgb.fit(Xtr_s, ytr)
yp_x = xgb.predict(Xte_s)
ypb_x = xgb.predict_proba(Xte_s)
acc_x = accuracy_score(yte, yp_x)
f1_x = f1_score(yte, yp_x, average='macro')
print(f'XGB Accuracy: {acc_x:.4f}, Macro F1: {f1_x:.4f}')
print()
print('Classification Report:')
print(classification_report(yte, yp_x, target_names=tnames, zero_division=0))

### 4.5 XGBoost + SMOTE

In [ ]:
print('Training XGBoost + SMOTE...')
sm = SMOTE(random_state=42)
Xtr_sm, ytr_sm = sm.fit_resample(Xtr_s, ytr)
print(f'SMOTE resampled train: {np.bincount(ytr_sm)}')

xgb_sm = XGBClassifier(learning_rate=0.1, max_depth=6, n_estimators=150,
                        random_state=42, eval_metric='mlogloss',
                        objective='multi:softprob', num_class=5)
xgb_sm.fit(Xtr_sm, ytr_sm)
yp_xsm = xgb_sm.predict(Xte_s)
ypb_xsm = xgb_sm.predict_proba(Xte_s)
acc_xsm = accuracy_score(yte, yp_xsm)
f1_xsm = f1_score(yte, yp_xsm, average='macro')
print(f'XGB+SMOTE Accuracy: {acc_xsm:.4f}, Macro F1: {f1_xsm:.4f}')
print()
print('Classification Report:')
print(classification_report(yte, yp_xsm, target_names=tnames, zero_division=0))

### 4.6 Model Performance Summary

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'XGBoost + SMOTE'],
    'Accuracy': [f'{acc_rf:.4f}', f'{acc_x:.4f}', f'{acc_xsm:.4f}'],
    'Macro F1': [f'{f1_rf:.4f}', f'{f1_x:.4f}', f'{f1_xsm:.4f}']
})
print(summary.to_string(index=False))

### 4.7 ROC Curves (One-vs-Rest)

In [ ]:
# ROC curves for each class
paper_auc = {'Fatal':0.96,'Major':0.88,'Minimal':0.96,'Minor':0.95,'Unknown':0.87}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
colors = ['steelblue', 'coral', 'forestgreen']
models_data = {'RF': ypb_rf, 'XGB': ypb_x, 'XGB+SMOTE': ypb_xsm}

for i, name in enumerate(tnames):
    yb = (yte == i).astype(int)
    if yb.sum() == 0:
        axes[i].text(0.5, 0.5, f'{name}\n(no data)', ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_title(name); continue
    for idx, (mn, pb) in enumerate(models_data.items()):
        prob = pb[:, i] if pb.shape[1] > i else np.zeros(len(yte))
        auc_val = roc_auc_score(yb, prob)
        fpr, tpr, _ = roc_curve(yb, prob)
        axes[i].plot(fpr, tpr, label=f'{mn} AUC={auc_val:.3f}', lw=2, color=colors[idx])
    axes[i].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[i].axhline(paper_auc.get(name, 0), color='gray', ls=':',
                    label=f'Paper={paper_auc.get(name, 0):.2f}')
    axes[i].legend(fontsize=8); axes[i].set_title(f'ROC — {name}', fontsize=12)
    axes[i].set_xlabel('FPR'); axes[i].set_ylabel('TPR')

for j in range(5, 6):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/toronto_reference/roc_curves.png', dpi=150)
plt.show()

In [ ]:
# AUC comparison with paper
print(f'{"Class":<12} {"Paper AUC":<10} {"Best AUC":<12} {"Model":<15}')
print('-' * 50)
for name in tnames:
    best_auc, best_model = 0, ''
    for mn, pb in models_data.items():
        i = sev_map[name]
        yb = (yte == i).astype(int)
        if yb.sum() == 0: continue
        prob = pb[:, i] if pb.shape[1] > i else np.zeros(len(yte))
        auc_val = roc_auc_score(yb, prob)
        if auc_val > best_auc:
            best_auc, best_model = auc_val, mn
    gap = best_auc - paper_auc.get(name, 0)
    print(f'{name:<12} {paper_auc.get(name, 0):<10.2f} {best_auc:.4f} (gap={gap:+.4f})  {best_model:<15}')

### 4.8 Feature Importance (XGBoost)

In [ ]:
# Feature importance
imp_df = pd.DataFrame({'feature': f_cols, 'importance': xgb.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(10, 8))
plt.barh(imp_df['feature'], imp_df['importance'], color='teal', edgecolor='black')
plt.title('Top 15 Feature Importances (XGBoost)', fontsize=13)
plt.xlabel('Importance'); plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig('outputs/toronto_reference/feature_importance.png', dpi=150)
plt.show()

# Summary

This notebook replicated the analysis from Boddepalli et al. (2026) for Toronto collision data:

1. **Data Cleaning & Aggregation** — 7,204 unique accidents from 19,754 person-level records
2. **EDA** — Yearly trends, neighbourhood hotspots, hourly patterns, driver behavior analysis, spatial clustering
3. **Cellular Automata Simulation** — Directional flow modeling, green time allocation, vehicle routing, delay prediction
4. **ML Severity Prediction** — Random Forest, XGBoost, XGBoost+SMOTE with temporal train/test split

All outputs saved to `outputs/toronto_reference/`.